In [1]:
import pandas as pd
import numpy as np
import os

RAW = '../data/raw/'
PROCESSED = '../data/processed/'
FINAL = '../data/final/'

#Cargamos el archivo sin cabecera para ver su estructura real
df_raw = pd.read_excel(RAW + 'comercio_IAE.xlsx', header=None)

#Cargamos el archivo sin cabecera para ver su estructura real
df_raw = pd.read_excel(RAW + 'comercio_IAE.xlsx', header=None)

print(f'Dimensiones: {df_raw.shape}')
print(f'\nPrimeras 10 filas, primeras 5 columnas:')
df_raw.iloc[:10, :5]

Dimensiones: (1154, 156)

Primeras 10 filas, primeras 5 columnas:


,0,1,2,3,4
0,D. Economía,NaN,NaN,NaN,NaN
1,Empresas y locales,NaN,NaN,NaN,NaN
2,Censo de locales y actividades,NaN,NaN,NaN,NaN
3,2.4.4.B. Actividades en locales Abiertos con T...,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,Ciudad de Madrid,01. Centro
5,NaN,NaN,NaN,Ciudad de Madrid,01. Centro
6,1 de julio de 2018,Ciudad de Madrid,Total,106.578,11.118
7,1 de julio de 2018,"A Agricultura, ganadería, silvicultura y pesca","A Agricultura, ganadería, silvicultura y pesca",12,1
8,1 de julio de 2018,"A Agricultura, ganadería, silvicultura y pesca","01 Agricultura, ganadería, caza y servicios re...",11,1
9,1 de julio de 2018,"A Agricultura, ganadería, silvicultura y pesca",02 Silvicultura y explotación forestal,1,0


In [2]:
#Revisamos las primeras filas completas para entender la estructura
print('Filas de cabecera (0-5):')
print(df_raw.iloc[:6, :4])

print('\nFilas de datos (6-10):')
print(df_raw.iloc[6:11, :4])

#Vemos cuántas filas y columnas totales
print(f'\nDimensiones totales: {df_raw.shape}')

#Revisamos fechass que hay en la columna 0
print('\nValores únicos en columna 0 (fechas):')
fechas = df_raw[0].dropna().unique()
for f in fechas:
    print(f' {f}')
    

Filas de cabecera (0-5):
                                                   0    1    2  \
0                                        D. Economía  NaN  NaN   
1                                 Empresas y locales  NaN  NaN   
2                     Censo de locales y actividades  NaN  NaN   
3  2.4.4.B. Actividades en locales Abiertos con T...  NaN  NaN   
4                                                NaN  NaN  NaN   
5                                                NaN  NaN  NaN   

                  3  
0               NaN  
1               NaN  
2               NaN  
3               NaN  
4  Ciudad de Madrid  
5  Ciudad de Madrid  

Filas de datos (6-10):
                     0                                               1  \
6   1 de julio de 2018                                Ciudad de Madrid   
7   1 de julio de 2018  A Agricultura, ganadería, silvicultura y pesca   
8   1 de julio de 2018  A Agricultura, ganadería, silvicultura y pesca   
9   1 de julio de 2018  A Agricultura,

In [3]:
#Los nombres de barrio están en las filas 4 y 5
#Fila 4 = distrito, Fila 5 = barrio
distritos = df_raw.iloc[4, 3:].tolist()
barrios = df_raw.iloc[5, 3:].tolist()

print(f'Número de columnas geográficas: {len(barrios)}')
print('\nPrimeros 10 barrios:')
for i, (d, b) in enumerate(zip(distritos[:10], barrios[:10])):
    print(f'  Col {i+3}: {d} | {b}')

Número de columnas geográficas: 153

Primeros 10 barrios:
  Col 3: Ciudad de Madrid | Ciudad de Madrid
  Col 4: 01. Centro | 01. Centro
  Col 5: 01. Centro | 011. Palacio
  Col 6: 01. Centro | 012. Embajadores
  Col 7: 01. Centro | 013. Cortes
  Col 8: 01. Centro | 014. Justicia
  Col 9: 01. Centro | 015. Universidad
  Col 10: 01. Centro | 016. Sol
  Col 11: 02. Arganzuela | 02. Arganzuela
  Col 12: 02. Arganzuela | 021. Imperial


In [4]:
#Identificamos qué columnas corresponden a barrios reales
#Un barrio real tiene código de 3 dígitos ('011. Palacio')
#Los subtotales de distrito tienen código de 2 dígitos ('01. Centro')

columnas_barrio = []
for i, barrio in enumerate(barrios):
    if barrio is not None and str(barrio).strip() != '':
        codigo = str(barrio).split('.')[0].strip()
        if len(codigo) == 3 and codigo.isdigit():
            columnas_barrio.append(i + 3)  #+3 porque empezamos en columna 3

print(f'Barrios reales encontrados: {len(columnas_barrio)}')
print('\nPrimeros 10:')
for col in columnas_barrio[:10]:
    print(f'  Col {col}: {barrios[col-3]}')

Barrios reales encontrados: 131

Primeros 10:
  Col 5: 011. Palacio
  Col 6: 012. Embajadores
  Col 7: 013. Cortes
  Col 8: 014. Justicia
  Col 9: 015. Universidad
  Col 10: 016. Sol
  Col 12: 021. Imperial
  Col 13: 022. Acacias
  Col 14: 023. Chopera
  Col 15: 024. Legazpi


In [5]:
#Fechas válidas que queremos conservar
fechas_validas = [
    '1 de julio de 2018',
    '1 de enero de 2019',
    '1 de julio de 2019',
    '1 de enero de 2020',
    '1 de julio de 2020',
    '1 de enero de 2021',
    '1 de julio de 2021',
    '1 de enero de 2022',
    '1 de julio de 2022',
    '1 de enero de 2023',
    '1 de julio de 2023'
]

#Filtramos solo filas con fecha válida y actividad de nivel superior (empieza por letra)
filas_validas = []
for idx, row in df_raw.iterrows():
    fecha = str(row[0]).strip()
    actividad = str(row[1]).strip()
    
    # Solo fechas válidas
    if fecha not in fechas_validas:
        continue
    
    # Solo actividades de nivel superior (una letra) o Total
    if actividad == 'Ciudad de Madrid' or (len(actividad) >= 1 and actividad[0].isupper() and actividad[1] == ' '):
        filas_validas.append(idx)

print(f'Filas válidas encontradas: {len(filas_validas)}')
print('\nMuestra de actividades encontradas:')
for idx in filas_validas[:15]:
    print(f'  {df_raw.iloc[idx, 0]} | {df_raw.iloc[idx, 1]}')

Filas válidas encontradas: 1133

Muestra de actividades encontradas:
  1 de julio de 2018 | Ciudad de Madrid
  1 de julio de 2018 | A Agricultura, ganadería, silvicultura y pesca
  1 de julio de 2018 | A Agricultura, ganadería, silvicultura y pesca
  1 de julio de 2018 | A Agricultura, ganadería, silvicultura y pesca
  1 de julio de 2018 | B Industrias extractivas
  1 de julio de 2018 | B Industrias extractivas
  1 de julio de 2018 | B Industrias extractivas
  1 de julio de 2018 | B Industrias extractivas
  1 de julio de 2018 | C Industria manufacturera
  1 de julio de 2018 | C Industria manufacturera
  1 de julio de 2018 | C Industria manufacturera
  1 de julio de 2018 | C Industria manufacturera
  1 de julio de 2018 | C Industria manufacturera
  1 de julio de 2018 | C Industria manufacturera
  1 de julio de 2018 | C Industria manufacturera


In [6]:
print('Valores únicos en columna 2 (subactividad):')
for idx in filas_validas[:20]:
    print(f'  col1: {df_raw.iloc[idx, 1][:40]} | col2: {df_raw.iloc[idx, 2]}')

Valores únicos en columna 2 (subactividad):
  col1: Ciudad de Madrid | col2: Total
  col1: A Agricultura, ganadería, silvicultura y | col2: A Agricultura, ganadería, silvicultura y pesca
  col1: A Agricultura, ganadería, silvicultura y | col2: 01 Agricultura, ganadería, caza y servicios relacionados con las mismas
  col1: A Agricultura, ganadería, silvicultura y | col2: 02 Silvicultura y explotación forestal
  col1: B Industrias extractivas | col2: B Industrias extractivas
  col1: B Industrias extractivas | col2: 06 Extracción de crudo de petróleo y gas natural
  col1: B Industrias extractivas | col2: 08 Otras industrias extractivas
  col1: B Industrias extractivas | col2: 09 Actividades de apoyo a las industrias extractivas
  col1: C Industria manufacturera | col2: C Industria manufacturera
  col1: C Industria manufacturera | col2: 10 Industria de la alimentación
  col1: C Industria manufacturera | col2: 11 Fabricación de bebidas
  col1: C Industria manufacturera | col2: 12 Industria 

In [7]:
#Filtramos solo filas donde col2 == col1 (son los totales por sector) o donde col2 == 'Total' (total ciudad)

filas_totales = []
for idx, row in df_raw.iterrows():
    fecha = str(row[0]).strip()
    col1 = str(row[1]).strip()
    col2 = str(row[2]).strip()
    
    if fecha not in fechas_validas:
        continue
    
    #Total ciudad o total de sector (col1 == col2)
    if col2 == 'Total' or col1 == col2:
        filas_totales.append(idx)

print(f'Filas totales encontradas: {len(filas_totales)}')
print(f'Esperado: {len(fechas_validas)} fechas x 22 actividades + 1 total = {len(fechas_validas) * 23}')
print('\nMuestra:')
for idx in filas_totales[:10]:
    print(f'  {df_raw.iloc[idx, 0]} | {df_raw.iloc[idx, 1][:40]}')

Filas totales encontradas: 242
Esperado: 11 fechas x 22 actividades + 1 total = 253

Muestra:
  1 de julio de 2018 | Ciudad de Madrid
  1 de julio de 2018 | A Agricultura, ganadería, silvicultura y
  1 de julio de 2018 | B Industrias extractivas
  1 de julio de 2018 | C Industria manufacturera
  1 de julio de 2018 | D Suministro de energía eléctrica, gas, 
  1 de julio de 2018 | E Suministro de agua, actividades de san
  1 de julio de 2018 | F Construcción
  1 de julio de 2018 | G Comercio al por mayor y al por menor; 
  1 de julio de 2018 | H Transporte y almacenamiento
  1 de julio de 2018 | I Hostelería


In [8]:
#Verificamos qué actividades hay por fecha
from collections import defaultdict

actividades_por_fecha = defaultdict(list)
for idx in filas_totales:
    fecha = str(df_raw.iloc[idx, 0]).strip()
    actividad = str(df_raw.iloc[idx, 1]).strip()
    actividades_por_fecha[fecha].append(actividad)

print('Actividades por fecha:')
for fecha, acts in sorted(actividades_por_fecha.items()):
    print(f'  {fecha}: {len(acts)} actividades')

#Ver qué fecha tiene menos y qué le falta
fecha_ref = '1 de julio de 2018'
print(f'\nActividades en {fecha_ref}:')
for a in actividades_por_fecha[fecha_ref]:
    print(f'  {a[:50]}')

Actividades por fecha:
  1 de enero de 2019: 22 actividades
  1 de enero de 2020: 22 actividades
  1 de enero de 2021: 22 actividades
  1 de enero de 2022: 22 actividades
  1 de enero de 2023: 22 actividades
  1 de julio de 2018: 22 actividades
  1 de julio de 2019: 22 actividades
  1 de julio de 2020: 22 actividades
  1 de julio de 2021: 22 actividades
  1 de julio de 2022: 22 actividades
  1 de julio de 2023: 22 actividades

Actividades en 1 de julio de 2018:
  Ciudad de Madrid
  A Agricultura, ganadería, silvicultura y pesca
  B Industrias extractivas
  C Industria manufacturera
  D Suministro de energía eléctrica, gas, vapor y ai
  E Suministro de agua, actividades de saneamiento, 
  F Construcción
  G Comercio al por mayor y al por menor; reparación
  H Transporte y almacenamiento
  I Hostelería
  J Información y comunicaciones
  K Actividades financieras y de seguros
  L Actividades inmobiliarias
  M Actividades profesionales, científicas y técnica
  N Actividades administrativas

In [9]:
#Construimos el dataframe limpio
#Extraemos los datos de las filas totales para los 131 barrios reales

registros = []

for idx in filas_totales:
    fecha = str(df_raw.iloc[idx, 0]).strip()
    actividad = str(df_raw.iloc[idx, 1]).strip()
    
    #Extraemos código de actividad (letra inicial) o Total
    if actividad == 'Ciudad de Madrid':
        cod_actividad = 'Total'
    else:
        cod_actividad = actividad[0]  #Primera letra: A, B, C...
    
    #Extraemos valores para cada barrio real
    for col in columnas_barrio:
        nombre_barrio = str(barrios[col - 3]).strip()
        valor = df_raw.iloc[idx, col]
        
        #Limpiamos valor numérico
        if pd.isna(valor):
            valor_num = 0
        else:
            valor_num = int(str(valor).replace('.', '').replace(',', ''))
        
        registros.append({
            'fecha': fecha,
            'cod_actividad': cod_actividad,
            'actividad': actividad,
            'barrio': nombre_barrio,
            'num_locales': valor_num
        })

df_comercio = pd.DataFrame(registros)

print(f'Dimensiones: {df_comercio.shape}')
print(f'\nMuestra:')
print(df_comercio.head(10))
print(f'\nActividades únicas: {df_comercio.cod_actividad.unique()}')

Dimensiones: (31702, 5)

Muestra:
                fecha cod_actividad         actividad            barrio  \
0  1 de julio de 2018         Total  Ciudad de Madrid      011. Palacio   
1  1 de julio de 2018         Total  Ciudad de Madrid  012. Embajadores   
2  1 de julio de 2018         Total  Ciudad de Madrid       013. Cortes   
3  1 de julio de 2018         Total  Ciudad de Madrid     014. Justicia   
4  1 de julio de 2018         Total  Ciudad de Madrid  015. Universidad   
5  1 de julio de 2018         Total  Ciudad de Madrid          016. Sol   
6  1 de julio de 2018         Total  Ciudad de Madrid     021. Imperial   
7  1 de julio de 2018         Total  Ciudad de Madrid      022. Acacias   
8  1 de julio de 2018         Total  Ciudad de Madrid      023. Chopera   
9  1 de julio de 2018         Total  Ciudad de Madrid      024. Legazpi   

   num_locales  
0         1661  
1         2813  
2         1119  
3         1806  
4         2247  
5         1472  
6          656  
7   

In [10]:
#Extraemos año y semestre de la fecha
def extraer_año(fecha):
    return int(fecha.split(' ')[-1])

def extraer_semestre(fecha):
    if 'julio' in fecha:
        return 2
    else:
        return 1

df_comercio['año'] = df_comercio['fecha'].apply(extraer_año)
df_comercio['semestre'] = df_comercio['fecha'].apply(extraer_semestre)

#Extraems código de barrio (primeros 3 dígitos)
df_comercio['cod_barrio'] = df_comercio['barrio'].apply(
    lambda x: x.split('.')[0].strip()
)

print('Muestra con nuevas columnas:')
print(df_comercio[['año', 'semestre', 'cod_barrio', 'barrio', 'cod_actividad', 'num_locales']].head(10))

print(f'\nAños disponibles: {sorted(df_comercio.año.unique())}')
print(f'Semestres: {df_comercio.semestre.unique()}')
print(f'Códigos de barrio muestra: {df_comercio.cod_barrio.unique()[:10]}')

Muestra con nuevas columnas:
    año  semestre cod_barrio            barrio cod_actividad  num_locales
0  2018         2        011      011. Palacio         Total         1661
1  2018         2        012  012. Embajadores         Total         2813
2  2018         2        013       013. Cortes         Total         1119
3  2018         2        014     014. Justicia         Total         1806
4  2018         2        015  015. Universidad         Total         2247
5  2018         2        016          016. Sol         Total         1472
6  2018         2        021     021. Imperial         Total          656
7  2018         2        022      022. Acacias         Total         1043
8  2018         2        023      023. Chopera         Total          663
9  2018         2        024      024. Legazpi         Total          387

Años disponibles: [2018, 2019, 2020, 2021, 2022, 2023]
Semestres: [2 1]
Códigos de barrio muestra: ['011' '012' '013' '014' '015' '016' '021' '022' '023' '0

In [11]:
#Trabajamos solo con semestre 1 para consistencia
#Excepción: 2018 solo tiene semestre 2, lo usamos igualmente
df_s1 = df_comercio[
    ((df_comercio['semestre'] == 1)) |
    ((df_comercio['año'] == 2018) & (df_comercio['semestre'] == 2))
].copy()

print(f'Registros tras filtro: {df_s1.shape}')
print(f'Años disponibles: {sorted(df_s1.año.unique())}')

#Pivotamos para tener una columna por actividad
df_pivot = df_s1.pivot_table(
    index=['año', 'cod_barrio', 'barrio'],
    columns='cod_actividad',
    values='num_locales',
    aggfunc='sum'
).reset_index()

df_pivot.columns.name = None
print(f'\nDimensiones pivot: {df_pivot.shape}')
print(df_pivot.head(5))

Registros tras filtro: (17292, 8)
Años disponibles: [2018, 2019, 2020, 2021, 2022, 2023]

Dimensiones pivot: (786, 24)
    año cod_barrio            barrio  A  B   C  D  E   F     G  ...   L   M  \
0  2018        011      011. Palacio  0  0  29  1  0  14   534  ...  15  39   
1  2018        012  012. Embajadores  0  0  51  0  0  25  1516  ...  11  43   
2  2018        013       013. Cortes  0  0  14  0  0   9   343  ...   5  19   
3  2018        014     014. Justicia  1  0  23  0  0  10   806  ...   8  29   
4  2018        015  015. Universidad  0  0  43  0  1  26   851  ...  14  51   

     N   O   P   Q   R    S  Total  U  
0  123  25  44  16  50  107   1661  2  
1  152   7  61  33  62  210   2813  3  
2   58  23  25  14  47   57   1119  5  
3   77  21  28  24  48  153   1806  0  
4  130  18  69  51  70  194   2247  5  

[5 rows x 24 columns]


In [12]:
#Calculamos indicadores comerciales como % sobre total de locales
#Evitamos división por cero
df_pivot['Total'] = df_pivot['Total'].replace(0, np.nan)

#Indicador 1: Hostelería y ocio (I + R)
df_pivot['pct_hosteleria_ocio'] = ((df_pivot['I'] + df_pivot['R']) / df_pivot['Total'] * 100).round(2)

#Indicador 2: Servicios profesionales y tech (M + J)
df_pivot['pct_servicios_prof'] = ((df_pivot['M'] + df_pivot['J']) / df_pivot['Total'] * 100).round(2)

#Indicador 3: Comercio tradicional y servicios básicos (G + Q + S)
df_pivot['pct_comercio_tradicional'] = ((df_pivot['G'] + df_pivot['Q'] + df_pivot['S']) / df_pivot['Total'] * 100).round(2)

#Indicador 4: Actividades inmobiliarias (L)
df_pivot['pct_inmobiliario'] = (df_pivot['L'] / df_pivot['Total'] * 100).round(2)
print('Muestra de indicadores calculados:')
cols_mostrar = ['año', 'cod_barrio', 'barrio', 'Total', 'pct_hosteleria_ocio', 'pct_servicios_prof','pct_comercio_tradicional', 'pct_inmobiliario']
print(df_pivot[cols_mostrar].head(10))

print('\nEstadísticas básicas 2023:')
df_2023 = df_pivot[df_pivot['año'] == 2023]
print(df_2023[['pct_hosteleria_ocio', 'pct_servicios_prof','pct_comercio_tradicional', 'pct_inmobiliario']].describe().round(2))

Muestra de indicadores calculados:
    año cod_barrio            barrio  Total  pct_hosteleria_ocio  \
0  2018        011      011. Palacio   1661                38.95   
1  2018        012  012. Embajadores   2813                20.41   
2  2018        013       013. Cortes   1119                43.61   
3  2018        014     014. Justicia   1806                31.45   
4  2018        015  015. Universidad   2247                31.78   
5  2018        016          016. Sol   1472                34.58   
6  2018        021     021. Imperial    656                20.58   
7  2018        022      022. Acacias   1043                19.65   
8  2018        023      023. Chopera    663                18.85   
9  2018        024      024. Legazpi    387                24.81   

   pct_servicios_prof  pct_comercio_tradicional  pct_inmobiliario  
0                3.79                     39.55              0.90  
1                3.70                     62.53              0.39  
2           

In [13]:
#Calculamos variación de hostelería/ocio entre 2018 y 2023
df_2018 = df_pivot[df_pivot['año'] == 2018][['cod_barrio', 'pct_hosteleria_ocio']].rename(columns={'pct_hosteleria_ocio': 'pct_hosteleria_2018'})

df_2023 = df_pivot[df_pivot['año'] == 2023][
    ['cod_barrio', 'barrio', 'pct_hosteleria_ocio', 'pct_servicios_prof', 'pct_comercio_tradicional', 'pct_inmobiliario']
    ].rename(columns={'pct_hosteleria_ocio': 'pct_hosteleria_2023'})

#Unimos 2018 y 2023
df_var = df_2023.merge(df_2018, on='cod_barrio')

#Variación en puntos porcentuales
df_var['var_hosteleria_1823'] = (df_var['pct_hosteleria_2023'] - df_var['pct_hosteleria_2018']).round(2)

print(f'Barrios con variación calculada: {len(df_var)}')
print('\nTop 10 barrios con mayor crecimiento de hostelería 2018-2023:')
print(df_var[['barrio', 'pct_hosteleria_2018', 'pct_hosteleria_2023', 'var_hosteleria_1823']].sort_values('var_hosteleria_1823', ascending=False)
      .head(10).to_string(index=False))

print('\nTop 10 barrios con mayor descenso de hostelería 2018-2023:')
print(df_var[['barrio', 'pct_hosteleria_2018','pct_hosteleria_2023', 'var_hosteleria_1823']].sort_values('var_hosteleria_1823', ascending=True)
      .head(10).to_string(index=False))

Barrios con variación calculada: 131

Top 10 barrios con mayor crecimiento de hostelería 2018-2023:
                   barrio  pct_hosteleria_2018  pct_hosteleria_2023  var_hosteleria_1823
        194. El Cañaveral                 0.00                28.74                28.74
               205. Rosas                21.63                31.44                 9.81
          215. Corralejos                41.54                46.89                 5.35
             122. Orcasur                20.17                24.43                 4.26
              202. Hellín                16.88                20.78                 3.90
      106. Cuatro Vientos                17.39                21.21                 3.82
093. Ciudad Universitaria                33.65                37.20                 3.55
            125. Moscardó                17.11                20.34                 3.23
          075. Ríos Rosas                23.87                26.95                 3.08
       035

In [14]:
#Seleccionamos columnas finales para el dataset de comercio
df_comercio_final = df_var[[
    'cod_barrio',
    'barrio', 
    'pct_hosteleria_2023',
    'pct_servicios_prof',
    'pct_comercio_tradicional',
    'pct_inmobiliario',
    'var_hosteleria_1823'
]].copy()

print(f'Dimensiones dataset comercio: {df_comercio_final.shape}')
print(f'Valores nulos:\n{df_comercio_final.isnull().sum()}')
print(f'\nMuestra final:')
print(df_comercio_final.head(10).to_string(index=False))

#Guardamos jnto al resto de datasets en processed
df_comercio_final.to_csv(PROCESSED + 'df_comercio.csv', index=False)

Dimensiones dataset comercio: (131, 7)
Valores nulos:
cod_barrio                  0
barrio                      0
pct_hosteleria_2023         0
pct_servicios_prof          0
pct_comercio_tradicional    0
pct_inmobiliario            0
var_hosteleria_1823         0
dtype: int64

Muestra final:
cod_barrio           barrio  pct_hosteleria_2023  pct_servicios_prof  pct_comercio_tradicional  pct_inmobiliario  var_hosteleria_1823
       011     011. Palacio                40.71                3.05                     37.97              1.18                 1.76
       012 012. Embajadores                22.82                3.69                     58.49              0.53                 2.41
       013      013. Cortes                44.61                2.63                     36.45              0.45                 1.00
       014    014. Justicia                31.72                1.92                     54.67              0.60                 0.27
       015 015. Universidad          